# Module 03 AI Workbook — solution and explanation

> Try the exercise yourself before reading this.

## The adversarial bug

[../adversarial_streams_buggy.cu](./adversarial_streams_buggy.cu) issues each
chunk's device→host copy on the wrong stream:

```cpp
square_plus_one<<<blocks, threads, 0, stream[s]>>>(d_in + offset, d_out + offset, count);
// BUG: D2H on stream[0], not stream[s]
cudaMemcpyAsync(h_out + offset, d_out + offset, cbytes, cudaMemcpyDeviceToHost, stream[0]);
```

**The rule:** operations issued to the *same* stream run in order; operations in
*different* streams have **no ordering guarantee** relative to each other. For
chunk `s > 0`, the kernel is on `stream[s]` but the copy is on `stream[0]`, so
the copy can start before the kernel finishes and read stale `d_out`.

**Why it often prints the right answer:** the program only inspects `h_out[0]`
and `h_out[N-1]`. Chunk 0's copy *is* on `stream[0]` (same as its kernel), so it
is correctly ordered; and the GPU is often fast enough that the race does not
manifest on a lightly loaded machine. That intermittency is exactly why a single
run and a two-element print are not a test.

**The fix:** put all three operations for a chunk on the same stream. The
corrected pipeline is [streams_solution.cu](solutions/streams_solution.cu):

```cpp
cudaMemcpyAsync(..., cudaMemcpyHostToDevice,   stream[s]);
square_plus_one<<<blocks, threads, 0,          stream[s]>>>(...);
cudaMemcpyAsync(..., cudaMemcpyDeviceToHost,   stream[s]);
```

## The overlap lesson

- **Pinned host memory** (`cudaMallocHost`) is required for `cudaMemcpyAsync` to
  be truly asynchronous; pageable memory silently serializes.
- With one stream per in-flight chunk, chunk `i+1`'s H2D copy can overlap chunk
  `i`'s kernel, and chunk `i`'s D2H can overlap chunk `i+1`'s compute. In the
  `nsys` timeline you should see the copy and compute rows interleave rather than
  stack end-to-end.
- If the workload is transfer-bound, overlap hides the copy time behind compute
  (or vice-versa); the best case is roughly the larger of the two totals, not
  their sum. Estimate that ceiling in Phase 3 before you measure.

## Mapping to the 5-phase loop

- **PREDICT:** name the cross-stream dependency as a risk from reading the code.
- **VERIFY:** a full-array check turns the intermittent race into a reliable
  `FAIL`. Two printed elements cannot.
- **DIAGNOSE:** the `nsys` timeline is the evidence — it shows both the overlap
  you achieved and, in the buggy version, the missing ordering.


## Run the worked solution

The cells below build and run the solution. Each should finish with `PASS`.

In [ ]:
!nvcc -O3 -arch=native solutions/streams_solution.cu -o sol && ./sol